In [ ]:
# prepare some melspec input from real audio

import soundfile
import torch
from vllm.model_executor.models.fastconformer_preprocessor import FilterbankFeatures

feats_extractor = FilterbankFeatures().cuda()

time_factor = 8
output_frames = 20


y, sr = soundfile.read("/home/vklimkov/workspace/vllm/vllm/example.wav", dtype="float32", always_2d=True)
assert sr == 16000
test_audio = torch.from_numpy(y.T).contiguous().cuda()  # shape [channels, samples]
# we need x8 frames, lets aim 160 frames of melspec
samples_num = feats_extractor.get_left_context_size() + feats_extractor.hop_length * output_frames * time_factor
test_audio = test_audio[:, :samples_num]

mel = feats_extractor(test_audio)[0].T.contiguous()
print(mel.shape)

import matplotlib.pyplot as plt
plt.imshow(mel.cpu().numpy().T, aspect="auto")
plt.colorbar()
plt.show()

In [ ]:

from vllm import SamplingParams
from vllm.engine.arg_utils import AsyncEngineArgs
from vllm.v1.engine.async_llm import AsyncLLM

import torch

type_str = "float32"
torch_type = getattr(torch, type_str)
engine_args = AsyncEngineArgs(
    model="toy_conv",
    dtype=type_str,
    max_model_len=256,
    gpu_memory_utilization=0.05,
    skip_tokenizer_init=True,
    enable_prefix_caching=False,
    #load_format="dummy",
    block_size=128,
    enforce_eager=True,
)
engine = AsyncLLM.from_engine_args(engine_args)
sampling_params = SamplingParams(max_tokens=64, skip_sampling=True)


request_id = "1"

# random input:
# T_target x Factor * Freq
freq = 80
x = mel.cpu().contiguous().clone().view(output_frames, time_factor * freq).contiguous()
print(x.shape)

prompt_len = 1
i = 0
inputs = {
    # dummy tokens
    "prompt_token_ids": [0] * prompt_len,
    # actual inpust to the model in prefill stage
    "custom_inputs": {
        "conv_input": x[i:i+prompt_len]  # 1 x 2 * 80
    }
}
i += prompt_len
outputs = []
async for output in engine.generate(inputs, sampling_params=sampling_params, request_id=request_id):
    conv_output = output.outputs[0].custom_outputs["conv_output"]  # T_target x Factor/2 x Freq/2 x Channels
    print(conv_output.shape)
    outputs.append(conv_output.clone())
    if i >= x.shape[0]:
        break
    inputs = {"conv_input": x[i:i+1]}
    i += 1
    await engine.append_request(request_id=request_id, custom_inputs=inputs)

vllm_res = torch.cat(outputs, dim=0).view(len(outputs), -1).contiguous().numpy()
print(vllm_res.shape)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from typing import Union


class CausalConv2D(nn.Conv2d):
    """
    A causal version of nn.Conv2d. It pads across frequency axis the same way the padding
    is implemented in nemo code, there is no padding across time axis.
    Instead we would feed extra left context that comes from the streaming buffer.
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int,
        stride: int = 1,
        padding: Union[str, int] = 0,
        dilation: int = 1,
        groups: int = 1,
        bias: bool = True,
        padding_mode: str = "zeros",
        device=None,
        dtype=None,
    ) -> None:
        assert not padding, "padding should be set to 0 or None for CausalConv2D."
        #self._left_padding = kernel_size - 1
        #self._right_padding = stride - 1
        self._left_padding = 1
        self._right_padding = 0

        padding = 0
        super(CausalConv2D, self).__init__(
            in_channels,
            out_channels,
            kernel_size,
            stride,
            padding,
            dilation,
            groups,
            bias,
            padding_mode,
            device,
            dtype,
        )

    def forward(
        self,
        x,  # B x CH x T x F
    ):
        # pad only frequencies
        #x = F.pad(x, pad=(self._left_padding, self._right_padding))
        x = F.pad(x, pad=(1, 0, 1, 0))
        x = super().forward(x)
        return x


class ConvSubsampling(nn.Module):
    """
    Minimal ConvSubsampling for dw_striding, causal mode.
    Configuration: subsampling_factor=8, feat_in=80, feat_out=512, conv_channels=256
    """

    def __init__(self, feat_in=80, feat_out=512, conv_channels=256):
        super(ConvSubsampling, self).__init__()

        # Fixed parameters for the specific configuration
        self.subsampling_factor = 8
        self._sampling_num = 3  # log2(8)
        self._stride = 2
        self._kernel_size = 3
        self._feat_in = feat_in
        self._feat_out = feat_out
        self._conv_channels = conv_channels

        # Build conv layers
        layers = []
        activation = nn.ReLU(inplace=True)

        # Layer 0: First causal conv (1 -> 256 channels)
        layers.append(
            CausalConv2D(
                in_channels=1,
                out_channels=conv_channels,
                kernel_size=self._kernel_size,
                stride=self._stride,
                padding=None,
            )
        )
        layers.append(activation)

        # Layers 2-7: Two iterations of (depthwise + pointwise + activation)
        for _ in range(self._sampling_num - 1):
            # Depthwise conv
            layers.append(
                CausalConv2D(
                    in_channels=conv_channels,
                    out_channels=conv_channels,
                    kernel_size=self._kernel_size,
                    stride=self._stride,
                    padding=None,
                    groups=conv_channels,
                )
            )
            # Pointwise conv
            layers.append(
                nn.Conv2d(
                    in_channels=conv_channels,
                    out_channels=conv_channels,
                    kernel_size=1,
                    stride=1,
                    padding=0,
                    groups=1,
                )
            )
            layers.append(activation)

        self.conv = nn.ModuleList(layers)

        # hard code the size across frequency axis after convolutions
        # this assumes `feat_in == 80`
        out_length = 11
        self.out = nn.Linear(conv_channels * out_length, feat_out)

    def get_left_context_size(self) -> int:
        """
        Computes how many frames of left context will be sliced off
        by the convolutional stack
        """
        resolution = 1
        total_left_context = 0
        for _ in range(self._sampling_num):
            total_left_context += (self._kernel_size - 1) * resolution
            resolution *= self._stride
        return total_left_context

    def forward(self, x):
        """
        Args:
            x: [B x 1 x T x F]
        Returns:
            [B x 256 x T x F]
        """
        # Transpose and add channel dimension: [B, F, T] -> [B, 1, T, F]
        #x = x.transpose(1, 2).unsqueeze(1)

        # Apply convolutions
        # make sure we get out_freq==11
        x = F.pad(x, pad=(8, 0))
        
        for conv in self.conv:
            x = conv(x)  # B x C x T x F
        print(x.shape)

        x = x.transpose(1, 2).flatten(start_dim=2)  # B x T x C*F
        x = self.out(x)

        return x

In [ ]:
from safetensors.torch import load_file

conv_subsampling = ConvSubsampling().cuda()

try:
    state_dict = load_file("toy_conv/model.safetensors")
    pre_enc_weights = {
        k[len("encoder.pre_encode.") :]: v
        for k, v in state_dict.items()
        if k.startswith("encoder.pre_encode.")
    }
    # load this weigths for pre encode,
    # filterbank weights are non-trainable and are re-created in the constructor
    conv_subsampling.load_state_dict(pre_enc_weights, strict=True)
    print("Weights loaded successfully.")
except Exception as e:
    print(f"Error loading weights: {e}")

print(mel.shape)
torch_mel = mel.unsqueeze(0).unsqueeze(0)  # 1 x 1 x time x freq
print(torch_mel.shape)
# pad to get 20 frames of output
#print(f"Padding time using left context: {conv_subsampling.get_left_context_size()}", flush=True)
#torch_mel = torch.nn.functional.pad(torch_mel, (0, 0, conv_subsampling.get_left_context_size(), 0))
#print(torch_mel.shape)

out = conv_subsampling(torch_mel)[0]  # time x dim
torch_res = out.detach().cpu().numpy()
print(torch_res.shape)




In [ ]:
plt.imshow(vllm_res.T, aspect="auto")
plt.colorbar()
# limit z-axis [-80, 80]
#plt.clim(-80, 80)
plt.show()
plt.imshow(torch_res.T, aspect="auto")
plt.colorbar()
#plt.clim(-80, 80)
plt.show()
diff = vllm_res.T - torch_res.T
print(diff.shape)
plt.imshow(diff[:, 0:], aspect="auto")
#plt.clim(-10, 10)
plt.colorbar()
plt.show()

import numpy as np
print(np.mean(np.abs(diff[:, 0:])))
print(diff[:, 0:])
